# 🦓 Playground 3: CycleGAN – Máy Biến Đổi Thế Giới (Unpaired Translation)
### Biến đổi phong cảnh: Ngựa $\leftrightarrow$ Ngựa vằn, Mùa hè $\leftrightarrow$ Mùa đông, Ảnh chụp $\leftrightarrow$ Tranh Van Gogh

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDanh1510/GAN-playground/blob/main/notebooks/03_cyclegan_world_transformer.ipynb)

---

## 🎯 Mục Tiêu Bài Học:
1. Hiểu cách CycleGAN biến đổi hình ảnh giữa 2 miền dữ liệu mà **không cần cặp ảnh tương ứng** (Unpaired Datasets).
2. Nắm vững nguyên lý **Vòng lặp Dịch thuật Khép kín (Cycle Consistency Loss)**: 
   $$\mathcal{L}_{cycle} = \|F(G(A)) - A\|_1 + \|G(F(B)) - B\|_1$$
3. Xây dựng 2 Máy tạo ($G_{A \to B}, F_{B \to A}$) và 2 Cảnh sát ($D_A, D_B$) thi đấu song song.

### 1. Cài đặt môi trường & Kiểm tra GPU

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Đang sử dụng thiết bị: {device}")
if torch.cuda.is_available():
    print(f"🔥 Tên GPU: {torch.cuda.get_device_name(0)}")

### 2. Kiến trúc Mạng Nơ-ron CycleGAN:
- **Residual Block Generator**: Giúp bảo toàn thông tin cấu trúc nền không đổi.
- **PatchGAN Discriminator**: Đánh giá độ chân thực của từng ô ảnh.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, 3, 1, 1, bias=False),
            nn.InstanceNorm2d(channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, 1, 1, bias=False),
            nn.InstanceNorm2d(channels)
        )
    def forward(self, x):
        return x + self.block(x)

class CycleGenerator(nn.Module):
    def __init__(self, in_c=3, out_c=3, n_res=3):
        super().__init__()
        model = [
            nn.Conv2d(in_c, 32, 7, 1, 3, bias=False),
            nn.InstanceNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, 2, 1, bias=False),
            nn.InstanceNorm2d(64),
            nn.ReLU(inplace=True)
        ]
        for _ in range(n_res):
            model.append(ResidualBlock(64))
        model += [
            nn.ConvTranspose2d(64, 32, 3, 2, 1, output_padding=1, bias=False),
            nn.InstanceNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, out_c, 7, 1, 3),
            nn.Tanh()
        ]
        self.net = nn.Sequential(*model)

    def forward(self, x):
        return self.net(x)

class CycleDiscriminator(nn.Module):
    def __init__(self, in_c=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_c, 32, 4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(32, 64, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(64),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 1, 4, 1, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

print("✓ Khởi tạo Kiến trúc CycleGenerator & CycleDiscriminator thành công!")

### 3. Tập dữ liệu không ghép cặp (Unpaired Data Batch: Ngựa thường vs Ngựa vằn)

In [ ]:
def get_unpaired_batch(batch_size=8, img_size=32):
    # Miền A: Khối tròn màu đơn sắc (Đại diện cho Ngựa thường)
    domain_A = torch.zeros(batch_size, 3, img_size, img_size)
    # Miền B: Khối có hoa văn kẻ sọc vằn (Đại diện cho Ngựa vằn)
    domain_B = torch.zeros(batch_size, 3, img_size, img_size)

    for i in range(batch_size):
        cy, cx = np.random.randint(10, 22, 2)
        y, x = np.ogrid[:img_size, :img_size]
        
        # Domain A
        mask_a = (x - cx)**2 + (y - cy)**2 <= 8**2
        domain_A[i, 0, mask_a] = 0.8
        domain_A[i, 1, mask_a] = 0.4
        domain_A[i, 2, mask_a] = 0.1

        # Domain B
        mask_b = (x - cx)**2 + (y - cy)**2 <= 8**2
        domain_B[i, :, mask_b] = 0.9
        stripes = (x % 4 == 0) & mask_b
        domain_B[i, :, stripes] = -0.9

    return domain_A * 2 - 1, domain_B

demo_A, demo_B = get_unpaired_batch(4)
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for i in range(4):
    axes[0, i].imshow(((demo_A[i].permute(1, 2, 0) + 1) / 2).clip(0, 1)); axes[0, i].set_title(f"Miền A {i+1} (Ngựa)"); axes[0, i].axis('off')
    axes[1, i].imshow(((demo_B[i].permute(1, 2, 0) + 1) / 2).clip(0, 1)); axes[1, i].set_title(f"Miền B {i+1} (Vằn)"); axes[1, i].axis('off')
plt.suptitle("Dữ Liệu Huấn Luyện Không Ghép Cặp (Unpaired Domains)")
plt.show()

### 4. Huấn luyện CycleGAN với Vòng Lặp Cycle Consistency Loss

In [ ]:
epochs = 35
batch_size = 8
lambda_cycle = 10.0
lambda_id = 5.0

G_AB = CycleGenerator().to(device)
F_BA = CycleGenerator().to(device)
D_A = CycleDiscriminator().to(device)
D_B = CycleDiscriminator().to(device)

criterion_gan = nn.BCELoss()
criterion_cycle = nn.L1Loss()
criterion_id = nn.L1Loss()

opt_G = optim.Adam(list(G_AB.parameters()) + list(F_BA.parameters()), lr=0.001, betas=(0.5, 0.999))
opt_D_A = optim.Adam(D_A.parameters(), lr=0.001, betas=(0.5, 0.999))
opt_D_B = optim.Adam(D_B.parameters(), lr=0.001, betas=(0.5, 0.999))

print("⚡ Đang bắt đầu huấn luyện CycleGAN...")
for epoch in range(1, epochs + 1):
    real_A, real_B = get_unpaired_batch(batch_size)
    real_A = real_A.to(device); real_B = real_B.to(device)

    # 1. Huấn luyện 2 Máy tạo G và F
    loss_id_A = criterion_id(F_BA(real_A), real_A) * lambda_id
    loss_id_B = criterion_id(G_AB(real_B), real_B) * lambda_id

    fake_B = G_AB(real_A); pred_fake_B = D_B(fake_B)
    loss_gan_AB = criterion_gan(pred_fake_B, torch.ones_like(pred_fake_B))

    fake_A = F_BA(real_B); pred_fake_A = D_A(fake_A)
    loss_gan_BA = criterion_gan(pred_fake_A, torch.ones_like(pred_fake_A))

    # Cycle Consistency Loss: A -> B -> A' và B -> A -> B'
    rec_A = F_BA(fake_B); loss_cycle_A = criterion_cycle(rec_A, real_A) * lambda_cycle
    rec_B = G_AB(fake_A); loss_cycle_B = criterion_cycle(rec_B, real_B) * lambda_cycle

    total_loss_G = loss_gan_AB + loss_gan_BA + loss_cycle_A + loss_cycle_B + loss_id_A + loss_id_B
    opt_G.zero_grad(); total_loss_G.backward(); opt_G.step()

    # 2. Huấn luyện 2 Cảnh sát D_A và D_B
    loss_D_A = (criterion_gan(D_A(real_A), torch.ones_like(pred_fake_A)) + criterion_gan(D_A(fake_A.detach()), torch.zeros_like(pred_fake_A))) / 2
    opt_D_A.zero_grad(); loss_D_A.backward(); opt_D_A.step()

    loss_D_B = (criterion_gan(D_B(real_B), torch.ones_like(pred_fake_B)) + criterion_gan(D_B(fake_B.detach()), torch.zeros_like(pred_fake_B))) / 2
    opt_D_B.zero_grad(); loss_D_B.backward(); opt_D_B.step()

    if epoch % 10 == 0 or epoch == epochs:
        print(f"Epoch [{epoch:02d}/{epochs}] | Cycle Loss: {(loss_cycle_A + loss_cycle_B).item():.4f} | Total G Loss: {total_loss_G.item():.4f}")

print("✓ Huấn luyện CycleGAN hoàn tất!")

### 5. Kiểm tra Chu Trình Khép Kín: $A \xrightarrow{G} B' \xrightarrow{F} A''$

In [ ]:
with torch.no_grad():
    test_A, _ = get_unpaired_batch(3)
    test_A = test_A.to(device)
    trans_B = G_AB(test_A).cpu()
    recon_A = F_BA(G_AB(test_A)).cpu()

fig, axes = plt.subplots(3, 3, figsize=(10, 9))
for i in range(3):
    axes[i, 0].imshow(((test_A[i].cpu().permute(1, 2, 0) + 1) / 2).clip(0, 1))
    axes[i, 0].set_title(f"Bước 1: Ảnh Gốc $A_{i+1}$"); axes[i, 0].axis('off')
    
    axes[i, 1].imshow(((trans_B[i].permute(1, 2, 0) + 1) / 2).clip(0, 1))
    axes[i, 1].set_title(f"Bước 2: G(A) Biến đổi (Ngựa vằn)"); axes[i, 1].axis('off')
    
    axes[i, 2].imshow(((recon_A[i].permute(1, 2, 0) + 1) / 2).clip(0, 1))
    axes[i, 2].set_title(f"Bước 3: F(G(A)) Tái tạo khép kín"); axes[i, 2].axis('off')

plt.suptitle("Kiểm Tra Chu Trình Khép Kín (Cycle Reconstruction Loop)", fontsize=14, fontweight='bold')
plt.show()